# Prediction Agreement Analysis

Consistency between different prediction methods: logit lens agreement,
head prediction agreement, component comparison, and stability.

## Why This Matters

Prediction agreement analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_agreement_analysis import (
    logit_lens_agreement, head_prediction_agreement,
    component_prediction_comparison, prediction_stability,
    prediction_agreement_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Lens Agreement

How early do layers agree on the final prediction?

In [ ]:
result = logit_lens_agreement(model, tokens, position=-1)
print(f"Final prediction: token {result['final_prediction']}")
print(f"Commit layer: {result['commit_layer']}")
print(f"Agreement: {result['agreement_fraction']:.2%}\n")
for p in result['per_layer']:
    flag = ' ✓' if p['agrees_with_final'] else ' ✗'
    print(f"  Layer {p['layer']}: pred={p['prediction']:3d}, "
          f"target_rank={p['final_token_rank']}{flag}")

## Head Prediction Agreement

Do attention heads agree or compete on the prediction?

In [ ]:
for layer in range(4):
    result = head_prediction_agreement(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: {result['n_unique_predictions']} unique preds, "
          f"agree={result['agreement_fraction']:.2f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: pred={h['prediction']:3d} (logit={h['max_logit']:.4f})")

## Component Prediction Comparison

Do attention and MLP agree at each layer?

In [ ]:
result = component_prediction_comparison(model, tokens, position=-1)
print(f"Agreement: {result['agreement_fraction']:.2%}")
print(f"Mean correlation: {result['mean_correlation']:.4f}\n")
for p in result['per_layer']:
    flag = ' [AGREE]' if p['agree'] else ' [DISAGREE]'
    print(f"  Layer {p['layer']}: attn→{p['attn_prediction']:3d}, "
          f"mlp→{p['mlp_prediction']:3d}, corr={p['logit_correlation']:.4f}{flag}")

## Prediction Stability

How often does the top prediction change between layers?

In [ ]:
result = prediction_stability(model, tokens, position=-1)
print(f"Total changes: {result['total_changes']}, stable: {result['is_stable']}\n")
for p in result['per_layer']:
    flag = ' ← CHANGED' if p['changed_from_previous'] else ''
    print(f"  Layer {p['layer']}: pred={p['prediction']:3d}, gap={p['logit_gap']:.4f}{flag}")

## Agreement Summary

Combined prediction agreement analysis.

In [ ]:
result = prediction_agreement_summary(model, tokens, position=-1)
print(f"Final prediction: token {result['final_prediction']}")
print(f"Commit layer: {result['commit_layer']}")
print(f"Layer agreement: {result['layer_agreement']:.2%}")
print(f"Changes: {result['total_changes']}, stable: {result['is_stable']}")
print(f"Attn-MLP agreement: {result['attn_mlp_agreement']:.2%}")
print(f"Attn-MLP correlation: {result['attn_mlp_correlation']:.4f}")